# Custom MCP Server with Streamable HTTP

In order to make this notebook work, you need to start the custom MCP server with streamable HTTP. This is how:

```bash
uv run server.py
```

In [ ]:
import os
import dotenv
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langfuse.langchain import CallbackHandler

# Load environment variables from .env file.
dotenv.load_dotenv()

# Initialize the Langfuse handler
langfuse_handler = CallbackHandler()

# Initialize the chat model.
model = init_chat_model(os.environ["LANGCHAIN_CHAT_MODEL_ANTHROPIC"])

Next we will create an MCP client that connects to two MCP servers. The first server `playwright` provides web browsing capabilities, while the second server `memory` offers access to a persistent memory store. We will also define a local file `memory.json` to store the agent's memory.

In [ ]:
# Create a multi-server MCP client with custom configurations.
client = MultiServerMCPClient(
    {
        "simple": {
            "transport": "streamable_http",
            "url": "http://localhost:8666/mcp/",
        }
    }
)

## Running the Agent

As we remember, it is easy to integrate tools into LangGraph agentic workflows and agents. MCP is no exception, as the MCP tools will be mapped automatically to LangChain tools. We can then invoke the agent with a user prompt, and it will decide which tools to use based on the context.

First, let us find out which tools are available from the MCP client:

In [ ]:
# Get the tools.
print("Fetching tools from the MCP client...")
tools = await client.get_tools()
for tool in tools:
    print(f"Tool: {tool.name} - {tool.description}")

This is a fine collection of tools. LangGraph's ReAct agent will be able to use them as needed.

In [ ]:
# Create the react agent without any tools.
agent = create_react_agent(
    model=model,
    tools=tools,
)

# Invoke the agent. This can take a while...
print("Invoking the agent...")
result = await agent.ainvoke(
    { "messages": [
        {
            "role": "user",
            "content": "Do something nice with the tools to show how they work.",
        }
    ]},
    config={"callbacks": [langfuse_handler], "recursion_limit": 50}
)
print("Agent response:")
for message in result["messages"]:
    message.pretty_print()

# Done.